In [3]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms


SEED = 42
BATCH_SIZE = 240
BASE_EPOCHS = 4
EARLY_STOP_MAX_EPOCHS = 7
EARLY_STOP_PATIENCE = 2

DATASET_NAME = "KMNIST"
DATA_ROOT = Path("./data")
ARTIFACTS_DIR = Path("./artifacts")
FIGURES_DIR = ARTIFACTS_DIR / "figures"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Torch version:", torch.__version__)

Device: cpu
Torch version: 2.10.0+cpu


In [4]:
import shutil

transform = transforms.Compose([
    transforms.ToTensor(),
])

kmnist_raw_dir = DATA_ROOT / "KMNIST" / "raw"
kmnist_raw_dir.mkdir(parents=True, exist_ok=True)

raw_names = [
    "train-images-idx3-ubyte",
    "train-labels-idx1-ubyte",
    "t10k-images-idx3-ubyte",
    "t10k-labels-idx1-ubyte",
]

for name in raw_names:
    src = DATA_ROOT / name
    dst = kmnist_raw_dir / name
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)

try:
    full_train = datasets.KMNIST(root=DATA_ROOT, train=True, transform=transform, download=False)
    test_ds = datasets.KMNIST(root=DATA_ROOT, train=False, transform=transform, download=False)
except Exception:
    full_train = datasets.KMNIST(root=DATA_ROOT, train=True, transform=transform, download=True)
    test_ds = datasets.KMNIST(root=DATA_ROOT, train=False, transform=transform, download=True)

train_size = int(0.8 * len(full_train))
val_size = len(full_train) - train_size

split_gen = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(full_train, [train_size, val_size], generator=split_gen)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

x_batch, y_batch = next(iter(train_loader))
print("Train size:", len(train_ds), "Val size:", len(val_ds), "Test size:", len(test_ds))
print("Batch x shape:", x_batch.shape, "Batch y shape:", y_batch.shape)
print("x min/max:", float(x_batch.min()), float(x_batch.max()))
print("Classes in batch:", torch.unique(y_batch).tolist())

Train size: 48000 Val size: 12000 Test size: 10000
Batch x shape: torch.Size([240, 1, 28, 28]) Batch y shape: torch.Size([240])
x min/max: 0.0 1.0
Classes in batch: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [5]:
class MLP(nn.Module):
    def __init__(self, input_dim=28 * 28, hidden_sizes=(512, 256), num_classes=10, dropout_p=0.0, use_bn=False):
        super().__init__()
        layers = [nn.Flatten()]
        prev = input_dim

        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            if use_bn:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_p > 0:
                layers.append(nn.Dropout(dropout_p))
            prev = h

        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def accuracy_from_logits(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_count += x.size(0)

    return total_loss / total_count, total_correct / total_count


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_count += x.size(0)

    return total_loss / total_count, total_correct / total_count


def model_summary(hidden_sizes, dropout_p, use_bn):
    return f"hidden={list(hidden_sizes)}, act=ReLU, dropout={dropout_p}, batchnorm={use_bn}"


def run_experiment(
    experiment_id,
    hidden_sizes=(512, 256),
    dropout_p=0.0,
    use_bn=False,
    optimizer_name="Adam",
    lr=1e-3,
    momentum=0.0,
    weight_decay=0.0,
    epochs=10,
    early_stopping=False,
    patience=4,
):
    model = MLP(hidden_sizes=hidden_sizes, dropout_p=dropout_p, use_bn=use_bn).to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name.lower() == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        momentum_val = 0.0
    elif optimizer_name.lower() == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
        momentum_val = momentum
    else:
        raise ValueError(f"Unknown optimizer: {optimizer_name}")

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }

    best_val_acc = -1.0
    best_val_loss = float("inf")
    best_state = None
    epochs_trained = 0
    no_improve_epochs = 0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)

        epochs_trained = epoch

        improved = (va_acc > best_val_acc) or (
            np.isclose(va_acc, best_val_acc) and va_loss < best_val_loss
        )

        if improved:
            best_val_acc = va_acc
            best_val_loss = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1

        if early_stopping and no_improve_epochs >= patience:
            print(f"[{experiment_id}] Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)

    result_row = {
        "experiment_id": experiment_id,
        "dataset": DATASET_NAME,
        "seed": SEED,
        "model_summary": model_summary(hidden_sizes, dropout_p, use_bn),
        "optimizer": optimizer_name,
        "lr": lr,
        "momentum": momentum_val,
        "weight_decay": weight_decay,
        "epochs_trained": epochs_trained,
        "best_val_accuracy": best_val_acc,
        "best_val_loss": best_val_loss,
    }

    return {
        "row": result_row,
        "history": history,
        "model": model,
        "best_state": best_state,
        "config": {
            "hidden_sizes": list(hidden_sizes),
            "dropout_p": dropout_p,
            "use_batchnorm": use_bn,
            "optimizer": optimizer_name,
            "lr": lr,
            "momentum": momentum_val,
            "weight_decay": weight_decay,
            "early_stopping": early_stopping,
            "patience": patience if early_stopping else None,
            "max_epochs": epochs,
        },
    }

In [6]:
results = []
all_runs = {}

base_arch = {
    "hidden_sizes": (256, 128),
    "dropout_p": 0.0,
    "use_bn": False,
}

# E1: base
E1 = run_experiment(
    experiment_id="E1",
    **base_arch,
    optimizer_name="Adam",
    lr=1e-3,
    weight_decay=0.0,
    epochs=BASE_EPOCHS,
)
results.append(E1["row"])
all_runs["E1"] = E1
print("E1 done, best val acc:", round(E1["row"]["best_val_accuracy"], 4))

# E2: Dropout
E2 = run_experiment(
    experiment_id="E2",
    hidden_sizes=base_arch["hidden_sizes"],
    dropout_p=0.25,
    use_bn=False,
    optimizer_name="Adam",
    lr=1e-3,
    weight_decay=0.0,
    epochs=BASE_EPOCHS,
)
results.append(E2["row"])
all_runs["E2"] = E2
print("E2 done, best val acc:", round(E2["row"]["best_val_accuracy"], 4))

# E3: BatchNorm
E3 = run_experiment(
    experiment_id="E3",
    hidden_sizes=base_arch["hidden_sizes"],
    dropout_p=0.0,
    use_bn=True,
    optimizer_name="Adam",
    lr=1e-3,
    weight_decay=0.0,
    epochs=BASE_EPOCHS,
)
results.append(E3["row"])
all_runs["E3"] = E3
print("E3 done, best val acc:", round(E3["row"]["best_val_accuracy"], 4))

# Для E4 выбираем лучшую архитектуру из E2/E3 по val_accuracy
best_candidate_id = max(["E2", "E3"], key=lambda k: all_runs[k]["row"]["best_val_accuracy"])
best_candidate = all_runs[best_candidate_id]
print("Best candidate for E4:", best_candidate_id)

cand_cfg = best_candidate["config"]

# E4: та же архитектура + EarlyStopping
E4 = run_experiment(
    experiment_id="E4",
    hidden_sizes=tuple(cand_cfg["hidden_sizes"]),
    dropout_p=cand_cfg["dropout_p"],
    use_bn=cand_cfg["use_batchnorm"],
    optimizer_name="Adam",
    lr=1e-3,
    weight_decay=0.0,
    epochs=EARLY_STOP_MAX_EPOCHS,
    early_stopping=True,
    patience=EARLY_STOP_PATIENCE,
)
results.append(E4["row"])
all_runs["E4"] = E4
print("E4 done, best val acc:", round(E4["row"]["best_val_accuracy"], 4))

# Финальная оценка лучшей модели на test (делаем 1 раз)
criterion = nn.CrossEntropyLoss()
best_model = E4["model"]
test_loss, test_acc = evaluate(best_model, test_loader, criterion, device)
print("Final test accuracy (E4):", round(test_acc, 4))


fixed_arch = {
    "hidden_sizes": tuple(E4["config"]["hidden_sizes"]),
    "dropout_p": E4["config"]["dropout_p"],
    "use_bn": E4["config"]["use_batchnorm"],
}

# O1: LR слишком большой
O1 = run_experiment(
    experiment_id="O1",
    **fixed_arch,
    optimizer_name="Adam",
    lr=5e-2,
    weight_decay=0.0,
    epochs=6,
)
results.append(O1["row"])
all_runs["O1"] = O1
print("O1 done, best val acc:", round(O1["row"]["best_val_accuracy"], 4))

# O2: LR слишком маленький
O2 = run_experiment(
    experiment_id="O2",
    **fixed_arch,
    optimizer_name="Adam",
    lr=5e-6,
    weight_decay=0.0,
    epochs=6,
)
results.append(O2["row"])
all_runs["O2"] = O2
print("O2 done, best val acc:", round(O2["row"]["best_val_accuracy"], 4))

# O3: SGD+momentum + weight_decay
O3 = run_experiment(
    experiment_id="O3",
    **fixed_arch,
    optimizer_name="SGD",
    lr=8e-3,
    momentum=0.9,
    weight_decay=2e-4,
    epochs=10,
)
results.append(O3["row"])
all_runs["O3"] = O3
print("O3 done, best val acc:", round(O3["row"]["best_val_accuracy"], 4))

runs_df = pd.DataFrame(results)
runs_df = runs_df[[
    "experiment_id",
    "dataset",
    "seed",
    "model_summary",
    "optimizer",
    "lr",
    "momentum",
    "weight_decay",
    "epochs_trained",
    "best_val_accuracy",
    "best_val_loss",
]]
runs_df.to_csv(ARTIFACTS_DIR / "runs.csv", index=False)

torch.save(E4["best_state"], ARTIFACTS_DIR / "best_model.pt")

best_config = {
    "hidden_sizes": E4["config"]["hidden_sizes"],
    "dropout_p": E4["config"]["dropout_p"],
    "use_batchnorm": E4["config"]["use_batchnorm"],
    "optimizer": E4["config"]["optimizer"],
    "lr": E4["config"]["lr"],
    "momentum": E4["config"]["momentum"],
    "weight_decay": E4["config"]["weight_decay"],
    "dataset": DATASET_NAME,
    "seed": SEED,
    "fast_mode": True,
    "epochs_trained": E4["row"]["epochs_trained"],
    "best_val_accuracy": E4["row"]["best_val_accuracy"],
    "best_val_loss": E4["row"]["best_val_loss"],
    "test_loss": test_loss,
    "test_accuracy": test_acc,
}


with open(ARTIFACTS_DIR / "best_config.json", "w", encoding="utf-8") as f:
    json.dump(best_config, f, ensure_ascii=False, indent=2)

h = E4["history"]
epochs_best = np.arange(1, len(h["train_loss"]) + 1)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_best, h["train_loss"], label="train_loss")
plt.plot(epochs_best, h["val_loss"], label="val_loss")
plt.title("E4 Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_best, h["train_acc"], label="train_acc")
plt.plot(epochs_best, h["val_acc"], label="val_acc")
plt.title("E4 Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "curves_best.png", dpi=150)
plt.close()

h1 = O1["history"]
h2 = O2["history"]
e1 = np.arange(1, len(h1["train_loss"]) + 1)
e2 = np.arange(1, len(h2["train_loss"]) + 1)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(e1, h1["train_loss"], label="O1 train")
plt.plot(e1, h1["val_loss"], label="O1 val")
plt.title("O1 (lr=5e-2): loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(e2, h2["train_loss"], label="O2 train")
plt.plot(e2, h2["val_loss"], label="O2 val")
plt.title("O2 (lr=5e-6): loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "curves_lr_extremes.png", dpi=150)
plt.close()

print("\nSaved artifacts:")
print("-", ARTIFACTS_DIR / "runs.csv")
print("-", ARTIFACTS_DIR / "best_model.pt")
print("-", ARTIFACTS_DIR / "best_config.json")
print("-", FIGURES_DIR / "curves_best.png")
print("-", FIGURES_DIR / "curves_lr_extremes.png")

runs_df

E1 done, best val acc: 0.939
E2 done, best val acc: 0.9421
E3 done, best val acc: 0.9499
Best candidate for E4: E3
[E4] Early stopping at epoch 7
E4 done, best val acc: 0.9563
Final test accuracy (E4): 0.8953
O1 done, best val acc: 0.9467
O2 done, best val acc: 0.7588
O3 done, best val acc: 0.9543

Saved artifacts:
- artifacts\runs.csv
- artifacts\best_model.pt
- artifacts\best_config.json
- artifacts\figures\curves_best.png
- artifacts\figures\curves_lr_extremes.png


,experiment_id,dataset,seed,model_summary,optimizer,lr,momentum,weight_decay,epochs_trained,best_val_accuracy,best_val_loss
0,E1,KMNIST,42,"hidden=[256, 128], act=ReLU, dropout=0.0, batc...",Adam,0.001000,0.0,0.0000,4,0.939000,0.205095
1,E2,KMNIST,42,"hidden=[256, 128], act=ReLU, dropout=0.25, bat...",Adam,0.001000,0.0,0.0000,4,0.942083,0.186683
2,E3,KMNIST,42,"hidden=[256, 128], act=ReLU, dropout=0.0, batc...",Adam,0.001000,0.0,0.0000,4,0.949917,0.165881
3,E4,KMNIST,42,"hidden=[256, 128], act=ReLU, dropout=0.0, batc...",Adam,0.001000,0.0,0.0000,7,0.956333,0.153636
4,O1,KMNIST,42,"hidden=[256, 128], act=ReLU, dropout=0.0, batc...",Adam,0.050000,0.0,0.0000,6,0.946667,0.195676
5,O2,KMNIST,42,"hidden=[256, 128], act=ReLU, dropout=0.0, batc...",Adam,0.000005,0.0,0.0000,6,0.758750,1.174556
6,O3,KMNIST,42,"hidden=[256, 128], act=ReLU, dropout=0.0, batc...",SGD,0.008000,0.9,0.0002,10,0.954333,0.153664


In [7]:
print("Best candidate from E2/E3:", best_candidate_id)
print("Best val accuracy (E4):", round(E4["row"]["best_val_accuracy"], 4))
print("Final test accuracy:", round(test_acc, 4))

runs_df_sorted = runs_df.sort_values("experiment_id").reset_index(drop=True)
runs_df_sorted

Best candidate from E2/E3: E3
Best val accuracy (E4): 0.9563
Final test accuracy: 0.8953


,experiment_id,dataset,seed,model_summary,optimizer,lr,momentum,weight_decay,epochs_trained,best_val_accuracy,best_val_loss
0,E1,KMNIST,42,"hidden=[256, 128], act=ReLU, dropout=0.0, batc...",Adam,0.001000,0.0,0.0000,4,0.939000,0.205095
1,E2,KMNIST,42,"hidden=[256, 128], act=ReLU, dropout=0.25, bat...",Adam,0.001000,0.0,0.0000,4,0.942083,0.186683
2,E3,KMNIST,42,"hidden=[256, 128], act=ReLU, dropout=0.0, batc...",Adam,0.001000,0.0,0.0000,4,0.949917,0.165881
3,E4,KMNIST,42,"hidden=[256, 128], act=ReLU, dropout=0.0, batc...",Adam,0.001000,0.0,0.0000,7,0.956333,0.153636
4,O1,KMNIST,42,"hidden=[256, 128], act=ReLU, dropout=0.0, batc...",Adam,0.050000,0.0,0.0000,6,0.946667,0.195676
5,O2,KMNIST,42,"hidden=[256, 128], act=ReLU, dropout=0.0, batc...",Adam,0.000005,0.0,0.0000,6,0.758750,1.174556
6,O3,KMNIST,42,"hidden=[256, 128], act=ReLU, dropout=0.0, batc...",SGD,0.008000,0.9,0.0002,10,0.954333,0.153664
